<a href="https://colab.research.google.com/github/siva123995/ProjectLLM/blob/main/LangGraphFirstAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langgraph langchain_google_genai langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import userdata
TAVILY_API_KEY=userdata.get('TavilyKey')
GEMINI_API_KEY = userdata.get('sivathede')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="models/gemini-3-flash-preview", google_api_key=GEMINI_API_KEY)
# model = ChatGoogleGenerativeAI(model="models/gemma-4-26b-a4b-it", google_api_key=GEMINI_API_KEY)

print(model.invoke("Hello World"))

content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EqQCCqECAQw51sdeluq+ZVo0la3J/YgDC5Cz8YgtRGDRdQruTHWNCh7B6JB6Omg+zpXRjQnlaIjwxSD2TeEzTEnu5O7u4AQArvItIJsuziORdk4agkbYVBZD+qRysoKEGO+p+mks1E530Wi7AXEKuATvt1IJFqgaFJSX5FuDQKtyrBBsF6MWY+Ln3ra+rgLxlxKhO6bXbOCWqLPY0en5oKG6hStl8TQdil0tYaLSgCtnwfn3VTU0qVf6zkYqDFFSa3cmt0Bybv4fbZXPS/ffPVWEcjrzrURavmrt8mkVogo99ArLtTzz8jgtimtdtKV/oZonOXUz8UU0sKJrwUP9UhMNrgnlIRQ+8X71IiUXNdbFUhiX3xmpUD5CI2Ru3zUkL3hGCb/idA=='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e2be6-048e-70f2-90e5-86ffcdfd1c3b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 3, 'output_tokens': 66, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 57}}


In [ ]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_community.tools import TavilySearchResults
from langgraph.graph import MessageGraph, END, StateGraph, START

search_tool = TavilySearchResults(tavily_api_key = TAVILY_API_KEY)

class AgentState(TypedDict):
  messages: Annotated[List[BaseMessage], add_messages]

def planning(state: AgentState):
  user_question = state['messages'][-1].content

  prompt = (
      "You're a helpful assistant, Based on user question, you need to decide if you need to search in the internet,"
      "If Yes, write a consice query find up-to-date information"
  )

  response = model.invoke([
      SystemMessage(content=prompt),
      HumanMessage(content=user_question)
  ])
  print(f'Response from planning:{response}')

  # Extract the text content from the model's response
  model_response_text = ""
  if isinstance(response.content, list) and len(response.content) > 0 and 'text' in response.content[0]:
      model_response_text = response.content[0]['text']
  elif isinstance(response.content, str):
      model_response_text = response.content
  else:
      model_response_text = str(response.content)

  final_message = {'messages': [AIMessage(content = model_response_text)]}
  return final_message

def call_search_tool(state: AgentState):
  # The last message's content should now be a string from the planning step
  user_query_for_search = state['messages'][-1].content
  search_results = search_tool.invoke(user_query_for_search)
  # TavilySearchResults.invoke() returns a string already
  final_message = {'messages': [AIMessage(content=str(search_results))]}
  return final_message


def responder(state: AgentState):
  search_result = state['messages'][-1].content
  user_question = state['messages'][0].content

  prompt = (
      "you're a helpful responder, Use the following search results to respond user question"
  )
  response = model.invoke(
      [HumanMessage(content = prompt),
      HumanMessage(content = f"User Question: {user_question} and search results: {search_result}")]
  )

  print(f'Response from Responder:{response}')

  # Similar to planning, ensure the content here is also a string
  model_response_text = ""
  if isinstance(response.content, list) and len(response.content) > 0 and 'text' in response.content[0]:
      model_response_text = response.content[0]['text']
  elif isinstance(response.content, str):
      model_response_text = response.content
  else:
      model_response_text = str(response.content)

  final_message = {'messages': [AIMessage(content = model_response_text)]}
  return final_message

workflow = StateGraph(AgentState)

workflow.add_node("planning", planning)
workflow.add_node("search", call_search_tool)
workflow.add_node("responder", responder)

workflow.add_edge(START, "planning")
workflow.add_edge("planning","search")
workflow.add_edge("search","responder")
workflow.add_edge("responder",END)

agent = workflow.compile()

inputs = {'messages': [HumanMessage(content = "what matches are today in ipl 2026?")]}
inputs = {'messages': [HumanMessage(content = "Who is the premier of Queensland?")]}
result = agent.invoke(inputs)
print(result['messages'][-1].content)

Response from planning:content=[{'type': 'text', 'text': 'Yes.\ncurrent Premier of Queensland', 'extras': {'signature': 'EqIGCp8GAQw51sd44eZWVo4xDs40WFl5G1qfjhj9o2lOXV2kEf7VAKRqJbzmYOb7JvOEsUsqKhGNmkwetpm//VAdk8ZK37kGk/2MZt/CTK9iMXzfXIDlQ6iPfhs5++kj1wstf7PpRoloNKLsTiNS6tLHI9IFoO7R02TGrRgh2ryhBBgIpi9ViGK1DtMYQTstJvfN2E8qS0q9Lpzb2nYzJHyEkkChH9k7hE+qStMEknhxdNzJFCPhIj5wF4iAJIFBeLU6q7eKQFMqNjV3pngcWZGJ76sTjkZXX4VfBSZ7gJAGDVHS2EKECDQeS+lhK6fLbNRiR7osaiGjkD3Fy4HX5APGNk85WkVopnWM6QTefrlw3ZHd9aSDZwiwiHld13QsQGafdk7ke6v3U7CYO74VsZ5yrhEZjSo+aSE23HfYuuYeB/JiuEGhuF33XWux6oqP/wyLVj6krWjCMRvbWjd3YcOqN7zBs1iw7q0ZvYe0KtRyVwzGUUNk8IObIT/Uf46Z3kUqauTUVLJjNev/b3Gnohyq45TwEi+m7aWZlkUwobR6OT275Q4ah6609y3tGJIZ1Dvm+bnoNBov+oOTpmGouYfFaniiLkbuAsSU0jTyT6hXAUrPKQLFhTqJg3N3hDuD7LKkem8Mg/25GdS0DKZ5EXgBA3rLV33peNVYXfnr94sAPOQF1bHBRI65t7nTNp+OTPk2RbPljk1Vyw5UnrihwCKi3ppax52pmh+39MEUyT1QZf88say9WtD0w1bWkTq7vFnn0YPDgIPBZjmobJ+ify7cla4m3EM8Wfc4UgaCUdcc37O/sQObOCFfzYtgJ/5XQbV/Dv+tX8Gib1xsoBlpKd0idzraBd1rgbJXoD6f6rDpSvy